<a href="https://colab.research.google.com/github/palldas/Legislative-NER-Detector/blob/main/Milestone1_Data_Prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install huggingface_hub pandas

In [ ]:
from huggingface_hub import hf_hub_download

zip_path = hf_hub_download(
    repo_id="iatpp/digitaldemocracy-2015-2018",
    filename="DH2024_Corpus_Release.zip",
    repo_type="dataset",
)

zip_path

'/root/.cache/huggingface/hub/datasets--iatpp--digitaldemocracy-2015-2018/snapshots/c7009b4f7afd7fe35c081f8c1e05c2b88db82321/DH2024_Corpus_Release.zip'

In [ ]:
import os

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"ZIP path: {zip_path}")
print(f"ZIP size: {zip_size_mb:.2f} MB")

ZIP path: /root/.cache/huggingface/hub/datasets--iatpp--digitaldemocracy-2015-2018/snapshots/c7009b4f7afd7fe35c081f8c1e05c2b88db82321/DH2024_Corpus_Release.zip
ZIP size: 344.53 MB


In [ ]:
import zipfile

# Choose the subset you want to demonstrate for Milestone 1
STATE = "CA"
SESSION = "2017-2018"

inner_path = f"DH2024_Corpus_Release/{STATE}/{SESSION}/CSV/speeches.csv"
print("Target file inside ZIP:", inner_path)

with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
    print("Total files in ZIP:", len(names))
    print("Found target file?", inner_path in names)

    # Show a few nearby paths for sanity
    matches = [p for p in names if f"DH2024_Corpus_Release/{STATE}/{SESSION}/CSV/" in p]
    print(f"Files under {STATE}/{SESSION}/CSV/:", len(matches))
    print("First 10 under that folder:")
    for p in matches[:10]:
        print("  ", p)

Target file inside ZIP: DH2024_Corpus_Release/CA/2017-2018/CSV/speeches.csv
Total files in ZIP: 119
Found target file? True
Files under CA/2017-2018/CSV/: 20
First 10 under that folder:
   DH2024_Corpus_Release/CA/2017-2018/CSV/
   DH2024_Corpus_Release/CA/2017-2018/CSV/committeeRosters.csv
   __MACOSX/DH2024_Corpus_Release/CA/2017-2018/CSV/._committeeRosters.csv
   DH2024_Corpus_Release/CA/2017-2018/CSV/legislature.csv
   __MACOSX/DH2024_Corpus_Release/CA/2017-2018/CSV/._legislature.csv
   DH2024_Corpus_Release/CA/2017-2018/CSV/people.csv
   __MACOSX/DH2024_Corpus_Release/CA/2017-2018/CSV/._people.csv
   DH2024_Corpus_Release/CA/2017-2018/CSV/hearings.csv
   __MACOSX/DH2024_Corpus_Release/CA/2017-2018/CSV/._hearings.csv
   DH2024_Corpus_Release/CA/2017-2018/CSV/videos.csv


In [ ]:
import pandas as pd
import zipfile

with zipfile.ZipFile(zip_path) as z:
    with z.open(inner_path) as f:
        cols = pd.read_csv(f, nrows=0).columns.tolist()  # default sep="," for CSV

print("Columns detected:")
print(cols)
print("Number of columns:", len(cols))

Columns detected:
['sid', 'pid', 'did', 'hid', 'bid', 'cid', 'hearingDate', 'session_year', 'hearingType', 'starting_vid', 'ending_vid', 'starting_time', 'ending_time', 'duration', 'last_name', 'first_name', 'text']
Number of columns: 17


In [ ]:
with zipfile.ZipFile(zip_path) as z:
    with z.open(inner_path) as f:
        # subtract 1 for header
        n_rows = sum(1 for _ in f) - 1

print("Row count (streamed):", n_rows)

Row count (streamed): 392565


In [ ]:
with zipfile.ZipFile(zip_path) as z:
    with z.open(inner_path) as f:
        sample = pd.read_csv(f, nrows=5)  # comma-separated
sample

,sid,pid,did,hid,bid,cid,hearingDate,session_year,hearingType,starting_vid,ending_vid,starting_time,ending_time,duration,last_name,first_name,text
0,298964,102,18254,10003,CA_201720180SR7,577,2016-12-05,2017,Regular,16523,16523,8,1106,996,De Leon,Kevin,If I can have the members please take their se...
1,298965,68,18254,10003,CA_201720180SR7,577,2016-12-05,2017,Regular,16523,16523,1106,1194,88,Nielsen,Jim,Pro Tem De Leon and former pro Tems and honora...
2,298966,102,18254,10003,CA_201720180SR7,577,2016-12-05,2017,Regular,16523,16523,1195,1204,9,De Leon,Kevin,Would the members elect and our guests beyond ...
3,298967,18504,18254,10003,CA_201720180SR7,577,2016-12-05,2017,Regular,16523,16523,1208,1289,81,Maryland Medow,Sister,Thank you. During this holy time of the year l...
4,298968,102,18254,10003,CA_201720180SR7,577,2016-12-05,2017,Regular,16523,16523,1289,1305,16,De Leon,Kevin,Amen. Will the color guard please present the ...


In [ ]:
import csv

wanted_cols = ["sid", "pid", "hid", "hearingDate", "session_year", "first_name", "last_name", "text"]
usecols = [c for c in wanted_cols if c in cols]
print("Using columns:", usecols)

with zipfile.ZipFile(zip_path) as z:
    with z.open(inner_path) as f:
        df = pd.read_csv(
            f,
            usecols=usecols,
            engine="python",
            on_bad_lines="skip"
        )

df["state"] = STATE
df["session"] = SESSION

print("Loaded rows:", len(df))
print("Loaded columns:", df.columns.tolist())
df.head(3)

Using columns: ['sid', 'pid', 'hid', 'hearingDate', 'session_year', 'first_name', 'last_name', 'text']
Loaded rows: 392358
Loaded columns: ['sid', 'pid', 'hid', 'hearingDate', 'session_year', 'last_name', 'first_name', 'text', 'state', 'session']


,sid,pid,hid,hearingDate,session_year,last_name,first_name,text,state,session
0,298964,102,10003,2016-12-05,2017,De Leon,Kevin,If I can have the members please take their se...,CA,2017-2018
1,298965,68,10003,2016-12-05,2017,Nielsen,Jim,Pro Tem De Leon and former pro Tems and honora...,CA,2017-2018
2,298966,102,10003,2016-12-05,2017,De Leon,Kevin,Would the members elect and our guests beyond ...,CA,2017-2018


In [ ]:
n_show = min(3, len(df))
for i in range(n_show):
    row = df.iloc[i]
    speaker = f"{row.get('first_name','')} {row.get('last_name','')}".strip()
    print("----")
    print("sid:", row.get("sid"))
    print("pid:", row.get("pid"))
    print("speaker:", speaker)
    print("text preview:", str(row.get("text",""))[:200])

----
sid: 298964
pid: 102
speaker: Kevin De Leon
text preview: If I can have the members please take their seats to our invited guests if they could be so kind enough up in the gallery, as well as behind the columns if you could please take your seats. Sergeants,
----
sid: 298965
pid: 68
speaker: Jim Nielsen
text preview: Pro Tem De Leon and former pro Tems and honorable colleagues, ladies and gentlemen, and Senators, that form of wonderful choral presentation, wasn't it a great statement about our nation? And what a w
----
sid: 298966
pid: 102
speaker: Kevin De Leon
text preview: Would the members elect and our guests beyond the rail and in the gallery please rise as we are lead in the prayer by our guest chaplain, Sister Libby Fernandez, Religious Sisters of Mary. After which


In [ ]:
df["text"] = df["text"].astype(str).str.strip()
df["speaker_name"] = (
    df.get("first_name", "").astype(str).str.strip()
    + " "
    + df.get("last_name", "").astype(str).str.strip()
).str.strip()

df[["sid", "pid", "speaker_name", "text"]].head(3)

,sid,pid,speaker_name,text
0,298964,102,Kevin De Leon,If I can have the members please take their se...
1,298965,68,Jim Nielsen,Pro Tem De Leon and former pro Tems and honora...
2,298966,102,Kevin De Leon,Would the members elect and our guests beyond ...


In [ ]:
out_path = f"/content/{STATE}_{SESSION}_speeches_project4_min.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)
df.head()

Saved: /content/CA_2017-2018_speeches_project4_min.csv


,sid,pid,hid,hearingDate,session_year,last_name,first_name,text,state,session,speaker_name
0,298964,102,10003,2016-12-05,2017,De Leon,Kevin,If I can have the members please take their se...,CA,2017-2018,Kevin De Leon
1,298965,68,10003,2016-12-05,2017,Nielsen,Jim,Pro Tem De Leon and former pro Tems and honora...,CA,2017-2018,Jim Nielsen
2,298966,102,10003,2016-12-05,2017,De Leon,Kevin,Would the members elect and our guests beyond ...,CA,2017-2018,Kevin De Leon
3,298967,18504,10003,2016-12-05,2017,Maryland Medow,Sister,Thank you. During this holy time of the year l...,CA,2017-2018,Sister Maryland Medow
4,298968,102,10003,2016-12-05,2017,De Leon,Kevin,Amen. Will the color guard please present the ...,CA,2017-2018,Kevin De Leon


In [ ]:
print("=== Milestone 1 Data Summary ===")
print("Dataset source: iatpp/digitaldemocracy-2015-2018 (Hugging Face)")
print(f"ZIP size: {zip_size_mb:.2f} MB")
print(f"Subset file: {inner_path}")
print(f"Subset row count (streamed): {n_rows}")
print(f"Subset columns available: {len(cols)}")
print(f"Columns used for Project 4: {usecols}")
print(f"Parsed into DataFrame rows: {len(df)} (after skipping any bad lines)")

=== Milestone 1 Data Summary ===
Dataset source: iatpp/digitaldemocracy-2015-2018 (Hugging Face)
ZIP size: 344.53 MB
Subset file: DH2024_Corpus_Release/CA/2017-2018/CSV/speeches.csv
Subset row count (streamed): 392565
Subset columns available: 17
Columns used for Project 4: ['sid', 'pid', 'hid', 'hearingDate', 'session_year', 'first_name', 'last_name', 'text']
Parsed into DataFrame rows: 392358 (after skipping any bad lines)
